# Phase 2 — Grid-Search Tuning, Custom-ENB5 and Classical ML Baselines

Follow-on to `teacher/Teacher_Screening_EarlyStopping.ipynb`, which screened 17 backbones with **frozen** ImageNet features. This notebook:

1. **Grid-searches the top 5 models** of that screen (by test macro F1), fine-tuning them the way `3. EfficientNetB5_osteo(Grid Search).ipynb` ("Custom-ENB5") trains.
2. **Retrains Custom-ENB5** under the same 300-epoch / early-stopping protocol, as the reference.
3. **Trains the 5 classical ML baselines** of `4. Base ML Models.ipynb`, with the same metrics, figures and result files.

### Models tuned (top 5 of the screening, by test macro F1)
| Rank | Model | Screened macro F1 (frozen backbone) |
|---|---|---|
| 1 | InceptionResNetV2 | 0.609 |
| 2 | ResNet50 | 0.605 |
| 3 | EfficientNetB7 | 0.593 |
| 4 | DenseNet201 | 0.590 |
| 5 | EfficientNetB4 | 0.582 |

### Training recipe — copied from notebook 3 (Custom-ENB5)
| Setting | Value |
|---|---|
| Input size | 224 × 224 for every model (notebook 3 uses 224 even for B5, whose native size is 456) |
| Preprocessing | the backbone's own `preprocess_input` only. **No BPHE**: notebook 3 defines `bphe_rgb` but never passes it to a generator |
| Augmentation (train only) | rotation 30°, width/height shift 0.2, shear 0.2, zoom 0.2, horizontal flip, brightness 0.8–1.2, fill `nearest` |
| Backbone | ImageNet weights, `include_top=False`, **every layer trainable** |
| Head | GAP → Dense 1024 (swish) → Dropout → Dense 512 (swish) → Dropout → Dense 1024 (swish) → Dropout → Dense 512 (swish) → Dropout → Dense 5 (softmax) |
| Loss / metric | categorical cross-entropy / accuracy |
| Epochs | up to **300**, with `EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)` |

**On freezing:** notebook 3 loads `EfficientNetB5(include_top=False)` and never freezes it, so its later `for layer in base_model.layers[-50:]: layer.trainable = True` is a no-op — those layers were already trainable. The model that scored 0.8948 therefore fine-tunes the **whole** backbone, and that is what is reproduced here.

### Grid — 36 combinations per model, the grid of notebook 2
| Parameter | Values |
|---|---|
| Learning rate | 0.0001, 0.001, 0.0003 |
| Dropout rate | 0.3, 0.5 |
| Optimizer | RMSprop, Adam, SGD (momentum 0.9) |
| Batch size | 16, 32 |

Every combination trains with the full 300-epoch / patience-10 protocol. The winner is the combination with the highest **validation macro F1**; validation accuracy is recorded as well, so the grid table can also be read in the format of the existing tuning table. The **test set is evaluated once per model**, on the winning combination only.

### Classical ML baselines (notebook 4)
Random Forest, XGBoost, CatBoost, SVM (RBF) and Gaussian Naive Bayes on BPHE-equalised flattened pixels (128 × 128, or 96 × 96 for Naive Bayes), with notebook 4's hyperparameters. As the analogue of "300 epochs with early stopping", XGBoost and CatBoost run **300 boosting rounds with early stopping on validation (patience 10)**; the other three fit once.

### Outputs — `results/` next to this notebook (`tuning/results/`)
```
results/<Model>/
    grid/<combo>.json     one file per grid combination (validation metrics, epochs, timing)
    grid_search.csv       all combinations, ranked
    grid_table_val_f1_macro.csv , grid_table_val_accuracy.csv    (LR × dropout rows, optimizer × batch columns)
    best/                 winning combination: metrics.json, history.csv, classification_report.*,
                          confusion_matrix.csv, test_predictions.csv, best.keras, figures/*.{pdf,png}
results/Custom-ENB5/ , results/RandomForest/ , XGBoost/ , CatBoost/ , SVM/ , NaiveBayes/
    metrics.json, history.csv (where applicable), classification_report.*, confusion_matrix.csv,
    test_predictions.csv, figures/*.{pdf,png}
results/_summary/
    summary.csv, advisor_table.csv, grid_best.csv, figures/*.{pdf,png}
```

### How to run
1. Open with the TensorFlow GPU kernel (WSL: TensorFlow 2.19 / Keras 3). Section 1.2 installs anything missing, including xgboost and catboost.
2. Check `DATA_DIR` in section 1.1.
3. *Run All.* **Everything is resumable**: each grid combination writes its JSON as soon as it finishes and is skipped on a re-run, so a crash or restart costs at most the run in progress.
4. Every TensorFlow step — each grid combination, each winner's test evaluation and Custom-ENB5 — runs in a **fresh subprocess** (`grid_worker.py`, written automatically next to this notebook; it re-runs this notebook's own setup cells). The notebook kernel itself never builds a model: it re-draws the saved figures inline. TensorFlow does not hand host memory back between runs — measured at roughly 340 MB per ResNet50-sized fine-tune — so 180 runs in one kernel would exhaust the machine. Training in a child process also means a crash, including out of memory, kills only that child. Set `WORKER_MODE` to `False` in section 1.1 to train in-process instead (useful when debugging).
5. If a combination fails, it is recorded as `failed` in its JSON and the grid moves on, so a single bad run cannot end a multi-day job. Delete that JSON to retry it.
6. **Budget roughly 40-70 hours** for the full grid (180 fine-tuning runs). Most runs stop after ~25 epochs; the SGD combinations are the slow ones. To split the work across sessions, set `MODELS` in section 1.1 to one or two models at a time.

## 1. Environment & Configuration
### 1.1 Settings
Edit here, or override any value with the matching `OSTEO_*` environment variable (used for quick test runs).

In [ ]:
import os

# Must be set before TensorFlow starts: allow GPU memory growth, hide TF info logs
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


def _env(key, default, cast=str):
    value = os.environ.get(key)
    return default if value in (None, "") else cast(value)


def _env_list(key, default):
    value = os.environ.get(key)
    return default if value in (None, "") else [v.strip() for v in value.split(",") if v.strip()]


CFG = dict(
    DATA_DIR=_env("OSTEO_DATA_DIR", "/mnt/d/FA019/new/SplittedDataset"),
    OUTPUT_DIR=_env("OSTEO_OUTPUT_DIR", "results"),        # -> tuning/results/<Model>/
    MODELS=_env("OSTEO_MODELS", "all"),                    # "all" or "Name1,Name2"
    RUN_CUSTOM_ENB5=_env("OSTEO_RUN_CUSTOM", 1, int) == 1,
    RUN_ML=_env("OSTEO_RUN_ML", 1, int) == 1,
    MAX_EPOCHS=_env("OSTEO_MAX_EPOCHS", 300, int),
    PATIENCE=_env("OSTEO_PATIENCE", 10, int),
    SKIP_EXISTING=_env("OSTEO_SKIP_EXISTING", 1, int) == 1,
    SAVE_WEIGHTS=_env("OSTEO_SAVE_WEIGHTS", 1, int) == 1,
    SEED=_env("OSTEO_SEED", 42, int),
    IMG_SIZE=_env("OSTEO_IMG_SIZE", 224, int),             # notebook 3 uses 224 for every model
    MONITOR="val_loss",
    SELECT_BY=_env("OSTEO_SELECT_BY", "val_f1_macro"),     # grid winner = highest validation macro F1
    FIT_VERBOSE=_env("OSTEO_FIT_VERBOSE", 0, int),         # 0 = compact progress, one line per LOG_EVERY epochs
    LOG_EVERY=_env("OSTEO_LOG_EVERY", 10, int),
    # Each grid combination trains in a fresh process: TensorFlow does not give host memory back between
    # runs, so 180 runs in one kernel would exhaust it. Set to 0 to train in-process (for debugging).
    WORKER_MODE=_env("OSTEO_WORKER_MODE", 1, int) == 1,
    NOTEBOOK=_env("OSTEO_NOTEBOOK", "Tuning_GridSearch_And_Baselines.ipynb"),
)

# The grid of notebook 2 / the parameter-tuning table
GRID = dict(
    learning_rate=[float(v) for v in _env_list("OSTEO_GRID_LR", ["0.0001", "0.001", "0.0003"])],
    dropout_rate=[float(v) for v in _env_list("OSTEO_GRID_DROPOUT", ["0.3", "0.5"])],
    optimizer=_env_list("OSTEO_GRID_OPTIMIZER", ["rmsprop", "adam", "sgd"]),
    batch_size=[int(v) for v in _env_list("OSTEO_GRID_BATCH", ["16", "32"])],
)
MAX_COMBOS = _env("OSTEO_MAX_COMBOS", 0, int)   # 0 = all; a small number shortens the grid for test runs

TUNED_ORDER = ["InceptionResNetV2", "ResNet50", "EfficientNetB7", "DenseNet201", "EfficientNetB4"]
ML_ORDER = ["RandomForest", "XGBoost", "CatBoost", "SVM", "NaiveBayes"]
CUSTOM_NAME = "Custom-ENB5"

_models = CFG["MODELS"]
if isinstance(_models, str):
    _models = TUNED_ORDER if _models.strip().lower() == "all" else [m.strip() for m in _models.split(",") if m.strip()]
_unknown = sorted(set(_models) - set(TUNED_ORDER))
assert not _unknown, f"Unknown model names: {_unknown}. Choose from {TUNED_ORDER}"
SELECTED = [m for m in TUNED_ORDER if m in _models]

N_COMBOS = len(GRID["learning_rate"]) * len(GRID["dropout_rate"]) * len(GRID["optimizer"]) * len(GRID["batch_size"])
if MAX_COMBOS:
    N_COMBOS = min(N_COMBOS, MAX_COMBOS)
print(f"Grid-search models : {SELECTED}")
print(f"Combinations/model : {N_COMBOS}  ->  {N_COMBOS * len(SELECTED)} fine-tuning runs")
print(f"Custom-ENB5        : {'yes' if CFG['RUN_CUSTOM_ENB5'] else 'no'}      Classical ML: {'yes' if CFG['RUN_ML'] else 'no'}")
print(f"Epochs             : up to {CFG['MAX_EPOCHS']}, early stopping on {CFG['MONITOR']}, patience {CFG['PATIENCE']}")
CFG

### 1.2 Packages
Installs anything missing into this kernel's environment, including `xgboost` and `catboost` for section 7. TensorFlow must already be present (installing it here could break its CUDA setup). The installed numpy / TensorFlow / PyTorch versions are pinned during the install so pip cannot upgrade or replace them.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

OPTIONAL_PACKAGES = {
    "numpy": "numpy<2.2", "pandas": "pandas", "cv2": "opencv-python-headless", "PIL": "Pillow", "scipy": "scipy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib", "seaborn": "seaborn", "scienceplots": "SciencePlots",
    "jinja2": "jinja2", "xgboost": "xgboost", "catboost": "catboost",
}
CORE_PACKAGES = {"tensorflow": "TensorFlow 2.16+ with GPU support (Keras 3)"}

missing_core = [name for name in CORE_PACKAGES if importlib.util.find_spec(name) is None]
if missing_core:
    raise ImportError(
        "Missing: " + ", ".join(CORE_PACKAGES[m] for m in missing_core) + ".\n"
        "Select the kernel that has it (e.g. the TensorFlow GPU kernel), or install it first, on WSL/Linux:\n"
        '    pip install "tensorflow[and-cuda]==2.19.0"')


def _installed_version(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


missing = [req for module, req in OPTIONAL_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    pins = [f"{dist}=={v}" for dist in ("numpy", "tensorflow", "torch", "torchvision") if (v := _installed_version(dist))]
    print("Installing    :", ", ".join(missing))
    print("Kept unchanged:", ", ".join(pins))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, *pins])
    importlib.invalidate_caches()
    print("Done.")
else:
    print("All required packages are installed.")

### 1.3 Imports

In [ ]:
import gc
import itertools
import json
import logging
import platform
import time
import warnings
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import PIL
from PIL import Image
from IPython.display import display

import tensorflow as tf

for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError as err:
        print("Memory growth not set:", err)

import keras
from tensorflow.keras.applications import (DenseNet201, EfficientNetB4, EfficientNetB5, EfficientNetB7,
                                           InceptionResNetV2, ResNet50,
                                           densenet, efficientnet, inception_resnet_v2, resnet50)
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam, RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, auc, classification_report, confusion_matrix, f1_score, log_loss,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore", message=".*PyDataset.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

### 1.4 Versions & GPU check

In [ ]:
assert keras.__version__.startswith("3"), f"Expected Keras 3 (TensorFlow 2.16+), found {keras.__version__}"

VERSIONS = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
            "numpy": np.__version__, "opencv": cv2.__version__, "pillow": PIL.__version__}
for dist in ("scikit-learn", "xgboost", "catboost"):
    try:
        VERSIONS[dist] = metadata.version(dist)
    except Exception:
        VERSIONS[dist] = "missing"
display(pd.Series(VERSIONS, name="version").to_frame())

TF_GPUS = tf.config.list_physical_devices("GPU")
with tf.device("/GPU:0" if TF_GPUS else "/CPU:0"):
    tf.reduce_sum(tf.matmul(tf.random.normal((256, 256)), tf.random.normal((256, 256)))).numpy()
print("TensorFlow GPU:", [g.name for g in TF_GPUS] or "none - running on CPU")
if not TF_GPUS:
    print("WARNING: no GPU visible. Fine-tuning the full grid on CPU is not practical.")

### 1.5 Paths

In [ ]:
DATA_DIR = Path(CFG["DATA_DIR"])
TRAIN_DIR, VAL_DIR, TEST_DIR = (DATA_DIR / split for split in ("train", "val", "test"))
for folder in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    assert folder.is_dir(), f"Missing data folder: {folder}"

OUT_DIR = Path(CFG["OUTPUT_DIR"])
SUMMARY_DIR = OUT_DIR / "_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Data   :", DATA_DIR.resolve())
print("Results:", OUT_DIR.resolve())

## 2. Figure Style
The same style as the screening notebook: SciencePlots (`science` + `ieee` + `no-latex`, with a plain-matplotlib fallback), bold serif text and colour-blind-safe Okabe–Ito colours. Every figure is saved as vector PDF plus 600-dpi PNG and shown inline at screen resolution.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ["png"]

try:
    import scienceplots  # noqa: F401  (registers the styles)
    plt.style.use(["science", "ieee", "no-latex"])
    SCIENCEPLOTS = True
except Exception:
    SCIENCEPLOTS = False

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.facecolor": "white",
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 16, "font.weight": "bold",
    "axes.labelsize": 18, "axes.labelweight": "bold", "axes.titlesize": 18, "axes.titleweight": "bold",
    "xtick.labelsize": 14, "ytick.labelsize": 14,
    "legend.fontsize": 13, "legend.title_fontsize": 14, "figure.titlesize": 20,
    "axes.linewidth": 1.5, "grid.linewidth": 1.0, "lines.linewidth": 2.4, "patch.linewidth": 1.2,
    "xtick.major.width": 1.5, "ytick.major.width": 1.5, "xtick.major.size": 6, "ytick.major.size": 6,
    "legend.frameon": True, "legend.framealpha": 0.95, "legend.edgecolor": "black", "legend.fancybox": True,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000"]
C_TRAIN, C_VAL, C_REF, C_CHANCE = "#0072B2", "#D55E00", "#222222", "#7F7F7F"
CLASS_COLORS = OKABE_ITO[:5]
KIND_COLORS = {"tuned": "#0072B2", "reference": "#222222", "ml": "#009E73"}


def save_fig(fig, stem, save=True):
    """Save as <stem>.pdf and <stem>.png (600 dpi), show inline, then close."""
    if save:
        stem = Path(stem)
        stem.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(f"{stem}.pdf")
        fig.savefig(f"{stem}.png", dpi=600)
    plt.show()
    plt.close(fig)


print("SciencePlots style:", "on" if SCIENCEPLOTS else "not available - using rcParams only")

## 3. Data Audit
Image counts per split and grade. Every file is loaded as 8-bit RGB, exactly as in the earlier notebooks.

In [ ]:
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
assert CLASS_NAMES == ["0", "1", "2", "3", "4"], f"Unexpected class folders: {CLASS_NAMES}"
NUM_CLASSES = len(CLASS_NAMES)
SPLIT_DIRS = {"train": TRAIN_DIR, "val": VAL_DIR, "test": TEST_DIR}

counts = pd.DataFrame({split: {c: sum(f.is_file() for f in (d / c).iterdir()) for c in CLASS_NAMES}
                       for split, d in SPLIT_DIRS.items()})
counts.index.name = "KL grade"
counts.loc["total"] = counts.sum()
display(counts)
N_TRAIN, N_VAL, N_TEST = (int(counts.loc["total", s]) for s in ("train", "val", "test"))

## 4. Shared Helpers
### 4.1 Data generators
Exactly the generators of notebook 3: the backbone's `preprocess_input`, the same augmentation for training, none for validation and test. `make_eval_generator` is the unshuffled version used for scoring.

In [ ]:
AUGMENTATION = dict(rotation_range=30, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2,
                    zoom_range=0.2, horizontal_flip=True, brightness_range=[0.8, 1.2], fill_mode="nearest")


def assert_class_order(*generators):
    expected = {c: i for i, c in enumerate(CLASS_NAMES)}
    for g in generators:
        assert g.class_indices == expected, f"Class order mismatch: {g.class_indices}"


def make_generators(preprocess_fn, batch_size, img_size=None, seed=None):
    """Train (augmented, shuffled) and validation (shuffled, as in notebook 3) generators."""
    size = (img_size or CFG["IMG_SIZE"],) * 2
    train_gen = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUGMENTATION).flow_from_directory(
        str(TRAIN_DIR), target_size=size, batch_size=batch_size, class_mode="categorical", seed=seed)
    val_gen = ImageDataGenerator(preprocessing_function=preprocess_fn).flow_from_directory(
        str(VAL_DIR), target_size=size, batch_size=batch_size, class_mode="categorical", seed=seed)
    assert_class_order(train_gen, val_gen)
    return train_gen, val_gen


def make_eval_generator(folder, preprocess_fn, batch_size, img_size=None):
    """Unshuffled, un-augmented generator, used for scoring validation and test."""
    gen = ImageDataGenerator(preprocessing_function=preprocess_fn).flow_from_directory(
        str(folder), target_size=(img_size or CFG["IMG_SIZE"],) * 2, batch_size=batch_size,
        class_mode="categorical", shuffle=False)
    assert_class_order(gen)
    return gen

### 4.2 Timing and epoch bookkeeping

In [ ]:
class ProgressEvery(Callback):
    """One compact line every `every` epochs (and for the first), instead of one line per epoch."""

    def __init__(self, every=10):
        super().__init__()
        self.every = max(1, every)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f"      epoch {epoch + 1:>3}: loss {logs.get('loss', float('nan')):.4f} "
                  f"acc {logs.get('accuracy', float('nan')):.4f} | "
                  f"val_loss {logs.get('val_loss', float('nan')):.4f} "
                  f"val_acc {logs.get('val_accuracy', float('nan')):.4f}", flush=True)


class EpochTimer(Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.perf_counter() - self._start)


def history_info(hist, best_idx, fit_time_s, stopped_early):
    times = hist["epoch_time_s"].to_numpy()
    return dict(
        epochs_run=len(hist), best_epoch=best_idx + 1, stopped_early=bool(stopped_early),
        best_val_loss=float(hist["val_loss"].iloc[best_idx]),
        val_accuracy_at_best=float(hist["val_accuracy"].iloc[best_idx]),
        train_accuracy_at_best=float(hist["accuracy"].iloc[best_idx]),
        final_val_accuracy=float(hist["val_accuracy"].iloc[-1]),
        max_val_accuracy=float(hist["val_accuracy"].max()),
        fit_time_s=float(fit_time_s), fit_time_min=float(fit_time_s) / 60,
        time_to_best_s=float(times[: best_idx + 1].sum()),
        epoch1_s=float(times[0]),
        median_epoch_s=float(np.median(times[1:])) if len(times) > 1 else float(times[0]),
    )

### 4.3 Metrics
Accuracy; precision, recall and F1 (weighted and macro); the per-grade classification report; the confusion matrix; and one-vs-rest ROC AUC per grade plus macro, weighted and micro averages. `prefix` is `"val"` or `"test"`.

In [ ]:
def compute_metrics(y_true, prob, prefix="test"):
    labels = list(range(NUM_CLASSES))
    y_pred = prob.argmax(axis=1)
    y_bin = label_binarize(y_true, classes=labels)
    auc_per_class = {}
    for i, c in enumerate(CLASS_NAMES):
        if y_bin[:, i].min() == y_bin[:, i].max():
            auc_per_class[c] = float("nan")
        else:
            fpr, tpr, _ = roc_curve(y_bin[:, i], prob[:, i])
            auc_per_class[c] = float(auc(fpr, tpr))
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), prob.ravel())
    kw = dict(zero_division=0)
    scalars = {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_precision_weighted": precision_score(y_true, y_pred, average="weighted", **kw),
        f"{prefix}_recall_weighted": recall_score(y_true, y_pred, average="weighted", **kw),
        f"{prefix}_f1_weighted": f1_score(y_true, y_pred, average="weighted", **kw),
        f"{prefix}_precision_macro": precision_score(y_true, y_pred, average="macro", **kw),
        f"{prefix}_recall_macro": recall_score(y_true, y_pred, average="macro", **kw),
        f"{prefix}_f1_macro": f1_score(y_true, y_pred, average="macro", **kw),
        f"{prefix}_auc_macro": float(np.nanmean(list(auc_per_class.values()))),
        f"{prefix}_auc_micro": auc(fpr_micro, tpr_micro),
    }
    try:
        scalars[f"{prefix}_auc_weighted"] = roc_auc_score(y_bin, prob, average="weighted")
    except ValueError:
        scalars[f"{prefix}_auc_weighted"] = float("nan")
    return dict(
        scalars={k: float(v) for k, v in scalars.items()},
        auc_per_class=auc_per_class,
        confusion_matrix=confusion_matrix(y_true, y_pred, labels=labels),
        report_text=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, digits=4, **kw),
        report_dict=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
                                          output_dict=True, **kw),
    )

### 4.4 Plots
Three figures per finished model: training and validation curves (the dotted line marks the restored best epoch), the test confusion matrix (counts and row percentages) and one-vs-rest ROC curves.

In [ ]:
def plot_curves(name, hist, best_epoch, stem, save=True):
    epochs = np.arange(1, len(hist) + 1)
    marker = "o" if len(hist) <= 30 else None
    panels = [(k, l) for k, l in (("accuracy", "Accuracy"), ("loss", "Loss"))
              if k in hist and not hist[k].isna().all()]        # boosting models record loss only
    fig, axes = plt.subplots(1, len(panels), figsize=(7.4 * len(panels), 5.6), squeeze=False)
    for ax, (key, label) in zip(axes[0], panels):
        ax.plot(epochs, hist[key], color=C_TRAIN, ls="-", lw=2.6, marker=marker, ms=5, label="Training")
        ax.plot(epochs, hist[f"val_{key}"], color=C_VAL, ls="-", lw=2.6, marker=marker, ms=5, label="Validation")
        ax.axvline(best_epoch, color=C_REF, ls=":", lw=2.2, label=f"Best epoch ({best_epoch})")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        ax.legend(loc="best")
    fig.suptitle(f"{name}: training and validation", fontweight="bold", y=1.04)
    save_fig(fig, stem, save)


def plot_confusion(name, cm, stem, save=True):
    cm = np.asarray(cm)
    pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100
    annot = np.array([[f"{n}\n({p:.1f}%)" for n, p in zip(row, prow)] for row, prow in zip(cm, pct)])
    fig, ax = plt.subplots(figsize=(8.2, 7.0))
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", square=True, linewidths=0.8, linecolor="white",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 13, "weight": "bold"},
                cbar_kws={"label": "Images"}, ax=ax)
    ax.minorticks_off()
    ax.tick_params(top=False, right=False)
    ax.set_xlabel("Predicted KL grade")
    ax.set_ylabel("True KL grade")
    ax.set_title(f"{name}: confusion matrix (test)")
    save_fig(fig, stem, save)


def plot_roc(name, y_true, prob, res, stem, save=True):
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7.6, 7.0))
    for i, c in enumerate(CLASS_NAMES):
        if y_bin[:, i].min() == y_bin[:, i].max():
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], prob[:, i])
        ax.plot(fpr, tpr, color=CLASS_COLORS[i], ls="-", lw=2.4, label=f"KL {c} (AUC = {res['auc_per_class'][c]:.3f})")
    fpr, tpr, _ = roc_curve(y_bin.ravel(), prob.ravel())
    ax.plot(fpr, tpr, color=C_REF, ls=":", lw=2.6, label=f"Micro-average (AUC = {res['test_auc_micro']:.3f})")
    ax.plot([0, 1], [0, 1], color=C_CHANCE, ls="--", lw=1.8, label="Chance")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_aspect("equal")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{name}: ROC (test)\nmacro AUC = {res['test_auc_macro']:.3f}")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", fontsize=11)
    save_fig(fig, stem, save)


def plot_train_val_bars(name, values, stem, save=True):
    """For models without an epoch history: train vs validation accuracy and log-loss."""
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.0))
    for ax, (key, label) in zip(axes, (("accuracy", "Accuracy"), ("log_loss", "Log loss"))):
        vals = [values[f"train_{key}"], values[f"val_{key}"]]
        bars = ax.bar(["Train", "Validation"], vals, color=[C_TRAIN, C_VAL], edgecolor="black", linewidth=1.0)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom",
                    fontsize=11, fontweight="bold")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(True, axis="y", alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
    fig.suptitle(f"{name}: training vs validation", fontweight="bold", y=1.04)
    save_fig(fig, stem, save)

### 4.5 Saving results
`finalize_run` writes every number and figure for one finished model. `metrics.json` is written last, so its presence means "this model is done" and a re-run skips it.

In [ ]:
def _json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Not JSON serialisable: {type(obj)}")


def write_json_atomic(path, data):
    tmp = Path(f"{path}.tmp")
    tmp.write_text(json.dumps(data, indent=2, default=_json_default))
    os.replace(tmp, path)


def print_result(name, res, report_path=None):
    print(f"\n{'=' * 70}\n{name}: test results\n{'=' * 70}")
    print(f"Test accuracy       : {res['test_accuracy']:.4f}")
    print(f"Precision weighted  : {res['test_precision_weighted']:.4f}   macro: {res['test_precision_macro']:.4f}")
    print(f"Recall    weighted  : {res['test_recall_weighted']:.4f}   macro: {res['test_recall_macro']:.4f}")
    print(f"F1 score  weighted  : {res['test_f1_weighted']:.4f}   macro: {res['test_f1_macro']:.4f}")
    print(f"ROC AUC   macro     : {res['test_auc_macro']:.4f}   micro: {res['test_auc_micro']:.4f}")
    if "val_f1_macro" in res:
        print(f"Validation macro F1 : {res['val_f1_macro']:.4f}   accuracy: {res.get('val_accuracy', float('nan')):.4f}")
    if "fit_time_min" in res:
        print(f"Training time       : {res['fit_time_min']:.1f} min")
    if report_path and Path(report_path).exists():
        print("\nClassification report (test):\n" + Path(report_path).read_text())


def finalize_run(name, out, info, y_true, prob, filenames, hist=None, curve_values=None):
    """Write metrics, CSVs and figures for one finished model; return its result dict."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    assert np.allclose(prob.sum(axis=1), 1.0, atol=1e-3), "Class probabilities do not sum to 1"
    m = compute_metrics(y_true, prob, "test")

    if hist is not None:
        hist.to_csv(out / "history.csv", index=False)
    pred = pd.DataFrame({"file": filenames, "y_true": y_true, "y_pred": prob.argmax(axis=1)})
    for i, c in enumerate(CLASS_NAMES):
        pred[f"prob_{c}"] = prob[:, i]
    pred.to_csv(out / "test_predictions.csv", index=False)
    (out / "classification_report.txt").write_text(m["report_text"])
    pd.DataFrame(m["report_dict"]).T.to_csv(out / "classification_report.csv")
    pd.DataFrame(m["confusion_matrix"], index=[f"true_{c}" for c in CLASS_NAMES],
                 columns=[f"pred_{c}" for c in CLASS_NAMES]).to_csv(out / "confusion_matrix.csv")

    res = {**info, **m["scalars"], "auc_per_class": m["auc_per_class"],
           "confusion_matrix": m["confusion_matrix"].tolist(),
           "finished_at": datetime.now().isoformat(timespec="seconds")}

    fig_dir = out / "figures"
    if hist is not None:
        plot_curves(name, hist, info.get("best_epoch", len(hist)), fig_dir / "training_curves")
    elif curve_values is not None:
        plot_train_val_bars(name, curve_values, fig_dir / "training_curves")
    plot_confusion(name, m["confusion_matrix"], fig_dir / "confusion_matrix")
    plot_roc(name, y_true, prob, {**res, **m}, fig_dir / "roc_curve")

    write_json_atomic(out / "metrics.json", res)
    print_result(name, res, out / "classification_report.txt")
    return res


def load_result(name, out, show=True):
    out = Path(out)
    res = json.loads((out / "metrics.json").read_text())
    if show:
        pred = pd.read_csv(out / "test_predictions.csv")
        prob = pred[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy()
        y_true = pred["y_true"].to_numpy()
        m = compute_metrics(y_true, prob, "test")
        hist_path = out / "history.csv"
        if hist_path.exists():
            hist = pd.read_csv(hist_path)
            plot_curves(name, hist, res.get("best_epoch", len(hist)), out / "figures" / "training_curves", save=False)
        plot_confusion(name, m["confusion_matrix"], out / "figures" / "confusion_matrix", save=False)
        plot_roc(name, y_true, prob, {**res, **m}, out / "figures" / "roc_curve", save=False)
        print_result(name, res, out / "classification_report.txt")
    return res


def cleanup_backend():
    keras.backend.clear_session()
    gc.collect()


RESULTS = {}

## 5. Grid Search — Top 5 Models
### 5.0 Model registry, builder and grid runner
`build_finetune_model` reproduces notebook 3's model: the ImageNet backbone with `include_top=False` and **no freezing**, followed by the 4-layer swish head with dropout.

Each combination trains with the full protocol (up to `MAX_EPOCHS`, early stopping on validation loss with `restore_best_weights`), then scores the restored weights on the validation set. Its numbers go to `grid/<combo>.json` immediately, so the search is resumable. Whenever a combination becomes the best so far, its weights and history are saved to `best/`; the **test set is only touched at the end**, once per model, by `evaluate_best`.

In [ ]:
TUNED_MODELS = {
    "InceptionResNetV2": dict(ctor=InceptionResNetV2, pre=inception_resnet_v2.preprocess_input, family="Inception"),
    "ResNet50": dict(ctor=ResNet50, pre=resnet50.preprocess_input, family="ResNet"),
    "EfficientNetB7": dict(ctor=EfficientNetB7, pre=efficientnet.preprocess_input, family="EfficientNet"),
    "DenseNet201": dict(ctor=DenseNet201, pre=densenet.preprocess_input, family="DenseNet"),
    "EfficientNetB4": dict(ctor=EfficientNetB4, pre=efficientnet.preprocess_input, family="EfficientNet"),
}
assert list(TUNED_MODELS) == TUNED_ORDER


def build_finetune_model(ctor, dropout_rate, img_size=None):
    """Notebook 3's model: unfrozen backbone + GAP + 1024/512/1024/512 swish head with dropout + softmax."""
    size = img_size or CFG["IMG_SIZE"]
    base_model = ctor(weights="imagenet", include_top=False, input_shape=(size, size, 3))
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    for units in (1024, 512, 1024, 512):
        x = Dense(units, activation="swish")(x)
        x = Dropout(dropout_rate)(x)
    predictions = Dense(NUM_CLASSES, activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    # Notebook 3 never freezes the backbone: every layer trains.
    assert all(layer.trainable for layer in base_model.layers), "Backbone should be fully trainable"
    return model


def make_optimizer(name, learning_rate):
    if name == "rmsprop":
        return RMSprop(learning_rate=learning_rate)
    if name == "adam":
        return Adam(learning_rate=learning_rate)
    if name == "sgd":
        return SGD(learning_rate=learning_rate, momentum=0.9)   # momentum 0.9, as in notebook 2
    raise ValueError(f"Unknown optimizer: {name}")


def combo_id(lr, dropout, optimizer, batch_size):
    return f"lr{lr:g}_do{dropout:g}_{optimizer}_bs{batch_size}"


def train_one(name, spec, lr, dropout, optimizer, batch_size, tag=""):
    """Train one configuration; return (model, history_df, info) with validation metrics of the restored weights."""
    keras.utils.set_random_seed(CFG["SEED"])
    train_gen, val_gen = make_generators(spec["pre"], batch_size, seed=CFG["SEED"])
    model = build_finetune_model(spec["ctor"], dropout)
    model.compile(optimizer=make_optimizer(optimizer, lr), loss="categorical_crossentropy", metrics=["accuracy"])

    early_stop = EarlyStopping(monitor=CFG["MONITOR"], patience=CFG["PATIENCE"], restore_best_weights=True, verbose=1)
    timer = EpochTimer()
    start = time.perf_counter()
    history = model.fit(train_gen, epochs=CFG["MAX_EPOCHS"], validation_data=val_gen,
                        callbacks=[early_stop, timer, ProgressEvery(CFG["LOG_EVERY"])],
                        verbose=CFG["FIT_VERBOSE"])
    fit_time = time.perf_counter() - start

    hist = pd.DataFrame(history.history)
    hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist["epoch_time_s"] = timer.epoch_times

    best_idx = int(getattr(early_stop, "best_epoch", hist["val_loss"].to_numpy().argmin()))
    if early_stop.best_weights is not None:
        model.set_weights(early_stop.best_weights)

    val_eval = make_eval_generator(VAL_DIR, spec["pre"], batch_size)
    val_prob = model.predict(val_eval, verbose=0)
    val_metrics = compute_metrics(val_eval.classes, val_prob, "val")["scalars"]

    info = dict(model=name, tag=tag, learning_rate=lr, dropout_rate=dropout, optimizer=optimizer,
                batch_size=batch_size, combo_id=combo_id(lr, dropout, optimizer, batch_size),
                img_size=CFG["IMG_SIZE"], max_epochs=CFG["MAX_EPOCHS"], patience=CFG["PATIENCE"],
                monitor=CFG["MONITOR"], seed=CFG["SEED"], recipe="notebook 3: full fine-tune + 4-dense head",
                params_total=int(model.count_params()), n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST,
                versions=VERSIONS)
    info.update(history_info(hist, best_idx, fit_time, early_stop.stopped_epoch > 0))
    info.update(val_metrics)
    return model, hist, info


WORKER_PATH = Path("grid_worker.py")
WORKER_SOURCE = r"""
# Run ONE step of the tuning notebook in a fresh process, then exit.
# Written by the tuning notebook; safe to delete. It executes the notebook's own setup cells, so the
# notebook stays the single source of truth for the recipe.
#   combo  NOTEBOOK MODEL CID LR DROPOUT OPTIMIZER BATCH    train one grid combination
#   best   NOTEBOOK MODEL                                   evaluate the winning combination on the test set
#   custom NOTEBOOK                                         train and evaluate Custom-ENB5
import sys
from pathlib import Path

import nbformat

MODE, NOTEBOOK = sys.argv[1], sys.argv[2]
SKIP_TAG = '# worker' + ': skip'   # assembled, so this line does not match itself

ns = {}
for cell in nbformat.read(NOTEBOOK, as_version=4).cells:
    if cell.cell_type != 'code':
        continue
    src = chr(10).join(l for l in cell.source.splitlines() if not l.lstrip().startswith(('%', '!')))
    if SKIP_TAG in src:
        continue
    exec(compile(src, '<notebook>', 'exec'), ns)

for needed in ('train_one', 'TUNED_MODELS', 'OUT_DIR', 'best_in_grid', 'write_json_atomic'):
    assert needed in ns, 'the worker skipped the notebook cell defining ' + needed

if MODE == 'combo':
    MODEL, CID = sys.argv[3], sys.argv[4]
    LR, DROPOUT, OPTIMIZER, BATCH = float(sys.argv[5]), float(sys.argv[6]), sys.argv[7], int(sys.argv[8])
    out = ns['OUT_DIR'] / MODEL
    model, hist, info = ns['train_one'](MODEL, ns['TUNED_MODELS'][MODEL], LR, DROPOUT, OPTIMIZER, BATCH,
                                        tag='grid')
    print('    -> val macro F1 %.4f | val accuracy %.4f | %d epochs (best %d) | %.1f min'
          % (info['val_f1_macro'], info['val_accuracy'], info['epochs_run'], info['best_epoch'],
             info['fit_time_min']), flush=True)
    select_by = ns['CFG']['SELECT_BY']
    previous_best = ns['best_in_grid'](out / 'grid')
    if previous_best is None or info[select_by] > previous_best[select_by]:
        (out / 'best').mkdir(parents=True, exist_ok=True)
        model.save(out / 'best' / 'best.keras')   # saved before the JSON, so the two cannot desynchronise
        hist.to_csv(out / 'best' / 'history.csv', index=False)
        ns['write_json_atomic'](out / 'best' / 'combo.json', info)
        print('    -> new best for ' + MODEL, flush=True)
    ns['write_json_atomic'](out / 'grid' / (CID + '.json'), info)
elif MODE == 'best':
    ns['evaluate_best'](sys.argv[3], ns['TUNED_MODELS'][sys.argv[3]])
elif MODE == 'custom':
    ns['train_custom_enb5']()
else:
    raise SystemExit('unknown worker mode: ' + MODE)
"""


def write_grid_worker():
    assert Path(CFG["NOTEBOOK"]).exists(), (
        f"Cannot find {CFG['NOTEBOOK']} in {Path.cwd()}. Set CFG['NOTEBOOK'] to this notebook's filename, "
        "or set CFG['WORKER_MODE'] to False to train in-process.")
    if not WORKER_PATH.exists() or WORKER_PATH.read_text() != WORKER_SOURCE:
        WORKER_PATH.write_text(WORKER_SOURCE)
    return WORKER_PATH


def run_in_worker(mode, *args):
    """Run one step (combo / best / custom) in a fresh process, streaming its output. Returns the exit code."""
    cmd = [sys.executable, str(write_grid_worker()), mode, CFG["NOTEBOOK"], *[str(a) for a in args]]
    env = {**os.environ, "MPLBACKEND": "Agg", "OSTEO_WORKER_MODE": "0"}
    noise = ("Found ", "Grid-search models", "Combinations/model", "Custom-ENB5 ", "Epochs  ", "Data   ",
             "Results:", "SciencePlots", "All required", "TensorFlow GPU", "Models to run", "Installing")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        line = line.rstrip()
        if line and not line.startswith(noise):
            print("   ", line[:300], flush=True)
    return proc.wait()


def best_in_grid(grid_dir):
    """Best finished combination so far, by CFG['SELECT_BY']."""
    rows = [json.loads(p.read_text()) for p in sorted(Path(grid_dir).glob("*.json"))]
    rows = [r for r in rows if CFG["SELECT_BY"] in r and not r.get("failed")]
    return max(rows, key=lambda r: r[CFG["SELECT_BY"]]) if rows else None


def grid_tables(rows, out):
    """Write the grid as CSVs shaped like the parameter-tuning table (LR x dropout rows, optimizer x batch columns)."""
    df = pd.DataFrame(rows)
    df = df.sort_values(CFG["SELECT_BY"], ascending=False)
    cols = ["combo_id", "learning_rate", "dropout_rate", "optimizer", "batch_size", "val_f1_macro", "val_accuracy",
            "val_f1_weighted", "best_val_loss", "epochs_run", "best_epoch", "stopped_early", "fit_time_min"]
    df.reindex(columns=cols).to_csv(out / "grid_search.csv", index=False)
    for metric in ("val_f1_macro", "val_accuracy"):
        table = df.pivot_table(index=["learning_rate", "dropout_rate"], columns=["optimizer", "batch_size"],
                               values=metric)
        table = table.reindex(index=pd.MultiIndex.from_product([GRID["learning_rate"], GRID["dropout_rate"]],
                                                               names=["learning_rate", "dropout_rate"]))
        wanted = [(o, b) for o in GRID["optimizer"] for b in GRID["batch_size"] if (o, b) in table.columns]
        table = table.reindex(columns=pd.MultiIndex.from_tuples(wanted, names=["optimizer", "batch_size"]))
        table.to_csv(out / f"grid_table_{metric}.csv")
        print(f"\n{metric} (rows: learning rate x dropout, columns: optimizer x batch size)")
        display(table.round(4))
    return df


def evaluate_best(name, spec):
    """Evaluate the winning combination on the test set: once per model."""
    out = OUT_DIR / name
    best_dir = out / "best"
    winner = best_in_grid(out / "grid")
    if winner is None:
        print(f"{name}: no finished combinations yet")
        return None
    saved = json.loads((best_dir / "combo.json").read_text()) if (best_dir / "combo.json").exists() else None

    if saved is None or saved["combo_id"] != winner["combo_id"] or not (best_dir / "best.keras").exists():
        print(f"{name}: re-training the winning combination {winner['combo_id']} to evaluate it")
        model, hist, info = train_one(name, spec, winner["learning_rate"], winner["dropout_rate"],
                                      winner["optimizer"], winner["batch_size"], tag="best")
        best_dir.mkdir(parents=True, exist_ok=True)
        model.save(best_dir / "best.keras")
        hist.to_csv(best_dir / "history.csv", index=False)
        write_json_atomic(best_dir / "combo.json", info)
    else:
        info = saved
        hist = pd.read_csv(best_dir / "history.csv")
        model = keras.models.load_model(best_dir / "best.keras")

    test_gen = make_eval_generator(TEST_DIR, spec["pre"], info["batch_size"])
    prob = model.predict(test_gen, verbose=0)
    res = finalize_run(name, best_dir, info, test_gen.classes, prob, test_gen.filenames, hist=hist)
    if not CFG["SAVE_WEIGHTS"]:
        (best_dir / "best.keras").unlink(missing_ok=True)
    del model
    cleanup_backend()
    return res


def run_grid(name):
    """Full grid for one model, resumable, then the single test evaluation of the winner."""
    if name not in SELECTED:
        print(f"{name}: not in MODELS - skipped")
        return None
    spec = TUNED_MODELS[name]
    out = OUT_DIR / name
    grid_dir = out / "grid"
    grid_dir.mkdir(parents=True, exist_ok=True)
    best_dir = out / "best"

    combos = list(itertools.product(GRID["learning_rate"], GRID["dropout_rate"], GRID["optimizer"],
                                    GRID["batch_size"]))
    if MAX_COMBOS:
        combos = combos[:MAX_COMBOS]

    for i, (lr, dropout, optimizer, batch_size) in enumerate(combos, start=1):
        cid = combo_id(lr, dropout, optimizer, batch_size)
        result_path = grid_dir / f"{cid}.json"
        if CFG["SKIP_EXISTING"] and result_path.exists():
            done = json.loads(result_path.read_text())
            if done.get("failed"):
                print(f"[{name} {i}/{len(combos)}] {cid}: failed earlier ({done['error'].split(':')[0]}) - "
                      f"skipped; delete grid/{cid}.json to retry it")
            else:
                print(f"[{name} {i}/{len(combos)}] {cid}: done earlier "
                      f"({CFG['SELECT_BY']}={done.get(CFG['SELECT_BY'], float('nan')):.4f}) - skipped")
            continue
        print(f"\n[{name} {i}/{len(combos)}] {cid}: lr={lr:g} dropout={dropout:g} {optimizer} batch={batch_size}")
        if CFG["WORKER_MODE"]:
            # A fresh process per combination: memory is reclaimed on exit, and a crash (even out of
            # memory) kills only the child. The worker writes the JSON, weights and history itself.
            code = run_in_worker("combo", name, cid, repr(float(lr)), repr(float(dropout)),
                                 optimizer, int(batch_size))
            if code != 0 or not result_path.exists():
                print(f"    -> FAILED (worker exited with code {code})")
                write_json_atomic(result_path, dict(model=name, combo_id=cid, learning_rate=lr,
                                                    dropout_rate=dropout, optimizer=optimizer,
                                                    batch_size=batch_size, failed=True,
                                                    error=f"WorkerError: worker exited with code {code}"))
            continue
        try:
            model, hist, info = train_one(name, spec, lr, dropout, optimizer, batch_size, tag="grid")
        except Exception as err:   # e.g. GPU out of memory: record it and keep the grid going
            print(f"    -> FAILED: {type(err).__name__}: {str(err)[:300]}")
            write_json_atomic(result_path, dict(model=name, combo_id=cid, learning_rate=lr, dropout_rate=dropout,
                                                optimizer=optimizer, batch_size=batch_size, failed=True,
                                                error=f"{type(err).__name__}: {str(err)[:500]}"))
            cleanup_backend()
            continue
        print(f"    -> val macro F1 {info['val_f1_macro']:.4f} | val accuracy {info['val_accuracy']:.4f} | "
              f"{info['epochs_run']} epochs (best {info['best_epoch']}) | {info['fit_time_min']:.1f} min")

        previous_best = best_in_grid(grid_dir)
        if previous_best is None or info[CFG["SELECT_BY"]] > previous_best[CFG["SELECT_BY"]]:
            best_dir.mkdir(parents=True, exist_ok=True)
            model.save(best_dir / "best.keras")          # saved before the JSON, so a crash cannot desynchronise them
            hist.to_csv(best_dir / "history.csv", index=False)
            write_json_atomic(best_dir / "combo.json", info)
            print(f"    -> new best for {name}")
        write_json_atomic(result_path, info)
        del model
        cleanup_backend()

    rows = [json.loads(p.read_text()) for p in sorted(grid_dir.glob("*.json"))]
    failed = [r for r in rows if r.get("failed")]
    rows = [r for r in rows if not r.get("failed")]
    if failed:
        print(f"{name}: {len(failed)} combination(s) failed and are excluded: "
              + ", ".join(f"{r['combo_id']} ({r['error'].split(':')[0]})" for r in failed))
    if not rows:
        print(f"{name}: no combination finished successfully")
        return None
    df = grid_tables(rows, out)
    winner = df.iloc[0]
    print(f"\n{name}: best combination -> {winner['combo_id']} "
          f"(val macro F1 {winner['val_f1_macro']:.4f}, val accuracy {winner['val_accuracy']:.4f})")
    best_dir = OUT_DIR / name / "best"
    evaluated = json.loads((best_dir / "metrics.json").read_text()) if (best_dir / "metrics.json").exists() else None
    if CFG["SKIP_EXISTING"] and evaluated and evaluated.get("combo_id") == winner["combo_id"]:
        print(f"{name}: winning combination already evaluated on the test set - loading those results")
        res = load_result(name, best_dir)
    elif CFG["WORKER_MODE"]:
        code = run_in_worker("best", name)
        if code != 0 or not (best_dir / "metrics.json").exists():
            print(f"{name}: evaluating the winner failed (worker exited with code {code})")
            return None
        res = load_result(name, best_dir)     # re-draw the figures the worker saved
    else:
        res = evaluate_best(name, spec)
    RESULTS[name] = res
    return res

### 5.1 InceptionResNetV2
Full grid, then one test evaluation of the winning combination.

In [ ]:
run_grid("InceptionResNetV2");  # worker: skip

### 5.2 ResNet50
Full grid, then one test evaluation of the winning combination.

In [ ]:
run_grid("ResNet50");  # worker: skip

### 5.3 EfficientNetB7
Full grid, then one test evaluation of the winning combination.

In [ ]:
run_grid("EfficientNetB7");  # worker: skip

### 5.4 DenseNet201
Full grid, then one test evaluation of the winning combination.

In [ ]:
run_grid("DenseNet201");  # worker: skip

### 5.5 EfficientNetB4
Full grid, then one test evaluation of the winning combination.

In [ ]:
run_grid("EfficientNetB4");  # worker: skip

## 6. Custom-ENB5 under the 300-epoch protocol
Notebook 3's configuration exactly — EfficientNetB5 at 224 × 224, unfrozen backbone, the 4-layer head with dropout 0.3, RMSprop at learning rate 1e-4, batch size 32 — retrained with the 300-epoch / patience-10 early stopping used everywhere in this notebook, so the reference is measured the same way as the tuned models.

In [ ]:
CUSTOM_SPEC = dict(ctor=EfficientNetB5, pre=efficientnet.preprocess_input, family="EfficientNet")
CUSTOM_PARAMS = dict(lr=0.0001, dropout=0.3, optimizer="rmsprop", batch_size=32)   # notebook 3's settings


def train_custom_enb5():
    """Train Custom-ENB5 with notebook 3's settings, then evaluate it on the test set."""
    out = OUT_DIR / CUSTOM_NAME
    model, hist, info = train_one(CUSTOM_NAME, CUSTOM_SPEC, CUSTOM_PARAMS["lr"], CUSTOM_PARAMS["dropout"],
                                  CUSTOM_PARAMS["optimizer"], CUSTOM_PARAMS["batch_size"], tag="reference")
    out.mkdir(parents=True, exist_ok=True)
    if CFG["SAVE_WEIGHTS"]:
        model.save(out / "best.keras")
    test_gen = make_eval_generator(TEST_DIR, CUSTOM_SPEC["pre"], CUSTOM_PARAMS["batch_size"])
    prob = model.predict(test_gen, verbose=0)
    res = finalize_run(CUSTOM_NAME, out, info, test_gen.classes, prob, test_gen.filenames, hist=hist)
    del model
    cleanup_backend()
    return res

In [ ]:
# worker: skip
if not CFG["RUN_CUSTOM_ENB5"]:
    print("Custom-ENB5: disabled in CFG - skipped")
elif CFG["SKIP_EXISTING"] and (OUT_DIR / CUSTOM_NAME / "metrics.json").exists():
    print(f"{CUSTOM_NAME}: finished results found - loading instead of retraining")
    RESULTS[CUSTOM_NAME] = load_result(CUSTOM_NAME, OUT_DIR / CUSTOM_NAME)
elif CFG["WORKER_MODE"]:
    code = run_in_worker("custom")
    if code == 0 and (OUT_DIR / CUSTOM_NAME / "metrics.json").exists():
        RESULTS[CUSTOM_NAME] = load_result(CUSTOM_NAME, OUT_DIR / CUSTOM_NAME)
    else:
        print(f"{CUSTOM_NAME}: worker exited with code {code} - see the output above")
else:
    RESULTS[CUSTOM_NAME] = train_custom_enb5()

## 7. Classical ML Baselines
### 7.0 Features
Notebook 4's pipeline: read with OpenCV, resize with `INTER_AREA`, equalise the Y channel (BPHE), scale, and flatten to one vector per image. 128 × 128 scaled to [-1, 1] for Random Forest, XGBoost, CatBoost and SVM; 96 × 96 scaled to [0, 1] for Naive Bayes.

In [ ]:
_FEATURE_CACHE = {}


def bphe_flat(path, img_size, scale):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, img_size, interpolation=cv2.INTER_AREA)
    y, cr, cb = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb))
    img = cv2.cvtColor(cv2.merge((cv2.equalizeHist(y), cr, cb)), cv2.COLOR_YCrCb2RGB).astype(np.float32)
    img = img / 127.5 - 1.0 if scale == "pm1" else img / 255.0
    return img.ravel()


def load_features(img_size=(128, 128), scale="pm1"):
    """X, y and file names per split, cached in memory."""
    key = (img_size, scale)
    if key in _FEATURE_CACHE:
        return _FEATURE_CACHE[key]
    data = {}
    for split, folder in SPLIT_DIRS.items():
        X, y, files = [], [], []
        for label, c in enumerate(CLASS_NAMES):
            for p in sorted((folder / c).iterdir()):
                if p.is_file():
                    X.append(bphe_flat(p, img_size, scale))
                    y.append(label)
                    files.append(f"{c}/{p.name}")
        data[split] = (np.stack(X), np.array(y), files)
        print(f"  {split}: {data[split][0].shape}")
    _FEATURE_CACHE[key] = data
    return data


def n_components(requested, n_samples, n_features):
    """PCA cannot ask for more components than samples or features (matters only on tiny test runs)."""
    return int(max(1, min(requested, n_samples - 1, n_features)))


def run_ml_model(name, fit_fn, img_size=(128, 128), scale="pm1"):
    """Train one classical model, evaluate it on validation and test, and write the usual results."""
    if not CFG["RUN_ML"]:
        print(f"{name}: classical ML disabled in CFG - skipped")
        return None
    out = OUT_DIR / name
    if CFG["SKIP_EXISTING"] and (out / "metrics.json").exists():
        print(f"{name}: finished results found - loading instead of retraining")
        RESULTS[name] = load_result(name, out)
        return RESULTS[name]

    print(f"\n{name}: loading features {img_size} ({scale})")
    data = load_features(img_size, scale)
    (X_train, y_train, _), (X_val, y_val, _), (X_test, y_test, test_files) = (data["train"], data["val"], data["test"])

    start = time.perf_counter()
    model, extra = fit_fn(X_train, y_train, X_val, y_val)
    fit_time = time.perf_counter() - start

    val_prob = model.predict_proba(X_val)
    train_prob = model.predict_proba(X_train)
    test_prob = model.predict_proba(X_test)
    val_metrics = compute_metrics(y_val, val_prob, "val")["scalars"]
    labels = list(range(NUM_CLASSES))
    curve_values = dict(train_accuracy=accuracy_score(y_train, train_prob.argmax(1)),
                        val_accuracy=accuracy_score(y_val, val_prob.argmax(1)),
                        train_log_loss=log_loss(y_train, train_prob, labels=labels),
                        val_log_loss=log_loss(y_val, val_prob, labels=labels))

    hist = extra.pop("history_df", None)
    # dict literal, not dict(**a, **b): curve_values and val_metrics share the key "val_accuracy"
    info = {"model": name, "kind": "ml", "img_size": list(img_size), "scale": scale,
            "n_features": int(X_train.shape[1]), "n_train": int(len(y_train)), "n_val": int(len(y_val)),
            "n_test": int(len(y_test)), "fit_time_s": float(fit_time), "fit_time_min": float(fit_time) / 60,
            "seed": CFG["SEED"], "versions": VERSIONS, **curve_values, **val_metrics, **extra}
    RESULTS[name] = finalize_run(name, out, info, y_test, test_prob, test_files,
                                 hist=hist, curve_values=curve_values)
    return RESULTS[name]

### 7.1 Random Forest
400 trees, `class_weight='balanced'`, on 128 × 128 BPHE pixels — notebook 4's settings. Trees have no epochs, so it fits once.

In [ ]:
def fit_random_forest(X_train, y_train, X_val, y_val):
    model = RandomForestClassifier(n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=CFG["SEED"])
    model.fit(X_train, y_train)
    return model, dict(n_estimators=400)


run_ml_model("RandomForest", fit_random_forest);  # worker: skip

### 7.2 XGBoost
Notebook 4's hyperparameters, with the analogue of the early-stopping protocol: up to 300 boosting rounds, stopping when validation multi-class log-loss has not improved for `PATIENCE` rounds.

In [ ]:
def fit_xgboost(X_train, y_train, X_val, y_val):
    from xgboost import XGBClassifier
    model = XGBClassifier(objective="multi:softprob", num_class=NUM_CLASSES, n_estimators=300, learning_rate=0.05,
                          max_depth=6, subsample=0.8, colsample_bytree=0.9, tree_method="hist",
                          random_state=CFG["SEED"], eval_metric="mlogloss",
                          early_stopping_rounds=CFG["PATIENCE"])
    model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
    ev = model.evals_result()
    hist = pd.DataFrame({"epoch": np.arange(1, len(ev["validation_0"]["mlogloss"]) + 1),
                         "loss": ev["validation_0"]["mlogloss"], "val_loss": ev["validation_1"]["mlogloss"]})
    hist["accuracy"] = np.nan
    hist["val_accuracy"] = np.nan
    best = int(getattr(model, "best_iteration", len(hist) - 1))
    return model, dict(epochs_run=len(hist), best_epoch=best + 1, stopped_early=len(hist) < 300,
                       best_val_loss=float(hist["val_loss"].iloc[best]), history_df=hist)


run_ml_model("XGBoost", fit_xgboost);  # worker: skip

### 7.3 CatBoost
Notebook 4's hyperparameters, with the overfitting detector set to the same patience: up to 300 iterations, stopping after `PATIENCE` iterations without improvement on validation.

In [ ]:
def fit_catboost(X_train, y_train, X_val, y_val):
    from catboost import CatBoostClassifier
    model = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=8, loss_function="MultiClass",
                               l2_leaf_reg=3, border_count=128, random_seed=CFG["SEED"], task_type="CPU",
                               od_type="Iter", od_wait=CFG["PATIENCE"], verbose=False, allow_writing_files=False)
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    ev = model.get_evals_result()
    train_loss = ev.get("learn", {}).get("MultiClass", [])
    val_loss = ev.get("validation", {}).get("MultiClass", [])
    hist = pd.DataFrame({"epoch": np.arange(1, len(val_loss) + 1), "loss": train_loss[: len(val_loss)],
                         "val_loss": val_loss})
    hist["accuracy"] = np.nan
    hist["val_accuracy"] = np.nan
    best = int(model.get_best_iteration())
    return model, dict(epochs_run=len(hist), best_epoch=best + 1, stopped_early=len(hist) < 300,
                       best_val_loss=float(model.get_best_score()["validation"]["MultiClass"]), history_df=hist)


run_ml_model("CatBoost", fit_catboost);  # worker: skip

### 7.4 SVM (RBF)
Notebook 4's pipeline: standardise, reduce to 300 whitened PCA components, then an RBF SVM with `C=10` and probability estimates.

In [ ]:
def fit_svm(X_train, y_train, X_val, y_val):
    k = n_components(300, X_train.shape[0], X_train.shape[1])
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("pca", PCA(n_components=k, svd_solver="randomized", whiten=True, random_state=CFG["SEED"])),
        ("svc", SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=CFG["SEED"])),
    ])
    model.fit(X_train, y_train)
    return model, dict(pca_components=k, C=10.0, kernel="rbf")


run_ml_model("SVM", fit_svm);  # worker: skip

### 7.5 Gaussian Naive Bayes
Notebook 4's pipeline: 96 × 96 pixels scaled to [0, 1], standardised, reduced to 200 PCA components, then Gaussian Naive Bayes.

In [ ]:
def fit_naive_bayes(X_train, y_train, X_val, y_val):
    k = n_components(200, X_train.shape[0], X_train.shape[1])
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("pca", PCA(n_components=k, svd_solver="randomized", random_state=CFG["SEED"])),
        ("gnb", GaussianNB()),
    ])
    model.fit(X_train, y_train)
    return model, dict(pca_components=k)


run_ml_model("NaiveBayes", fit_naive_bayes, img_size=(96, 96), scale="01");  # worker: skip

## 8. Summary
### 8.1 Results table
Every finished model: the five tuned backbones (with their winning hyperparameters), Custom-ENB5 and the classical baselines. Validation numbers come from the restored best weights; test numbers come from the single test evaluation.

In [ ]:
# worker: skip
def collect_results():
    rows = []
    for name in TUNED_ORDER:
        path = OUT_DIR / name / "best" / "metrics.json"
        if path.exists():
            rows.append({**json.loads(path.read_text()), "kind": "tuned"})
    path = OUT_DIR / CUSTOM_NAME / "metrics.json"
    if path.exists():
        rows.append({**json.loads(path.read_text()), "kind": "reference"})
    for name in ML_ORDER:
        path = OUT_DIR / name / "metrics.json"
        if path.exists():
            rows.append({**json.loads(path.read_text()), "kind": "ml"})
    return rows


records = collect_results()
print(f"Finished models: {len(records)}")
COLUMNS = ["model", "kind", "optimizer", "learning_rate", "dropout_rate", "batch_size", "epochs_run", "best_epoch",
           "stopped_early", "val_accuracy", "val_f1_macro", "test_accuracy", "test_precision_weighted",
           "test_recall_weighted", "test_f1_weighted", "test_precision_macro", "test_recall_macro", "test_f1_macro",
           "test_auc_macro", "fit_time_min", "params_total"]
summary = pd.DataFrame(records).reindex(columns=COLUMNS)
summary.to_csv(SUMMARY_DIR / "summary.csv", index=False)

advisor = summary[["model", "val_accuracy", "test_accuracy", "test_f1_macro", "fit_time_min"]].rename(
    columns={"model": "Model Name", "val_accuracy": "Validation Accuracy", "test_accuracy": "Test Accuracy",
             "test_f1_macro": "Test Macro F1", "fit_time_min": "Training Time (min)"})
advisor.to_csv(SUMMARY_DIR / "advisor_table.csv", index=False)

best_rows = []
for name in TUNED_ORDER:
    path = OUT_DIR / name / "grid_search.csv"
    if path.exists():
        g = pd.read_csv(path)
        best_rows.append({"model": name, **g.iloc[0].to_dict(), "combos_done": len(g)})
if best_rows:
    pd.DataFrame(best_rows).to_csv(SUMMARY_DIR / "grid_best.csv", index=False)

shown = summary.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
metric_cols = ["val_accuracy", "val_f1_macro", "test_accuracy", "test_f1_weighted", "test_f1_macro", "test_auc_macro"]
try:
    display(shown.style.format(precision=4, na_rep="-").highlight_max(subset=metric_cols, color="#cfe8ff"))
except Exception:
    display(shown.round(4))
print("Saved:", SUMMARY_DIR / "summary.csv", "and", SUMMARY_DIR / "advisor_table.csv")

### 8.2 Figures
Test metrics per model, training time, and a heat map of each grid (validation macro F1 over the 36 combinations).

In [ ]:
# worker: skip
FIG_SUM = SUMMARY_DIR / "figures"
df = summary.dropna(subset=["test_f1_macro"]).copy()

if df.empty:
    print("No finished models yet.")
else:
    names, x = df["model"].tolist(), np.arange(len(df))
    metrics = [("test_accuracy", "Accuracy"), ("test_f1_macro", "Macro F1"),
               ("test_f1_weighted", "Weighted F1"), ("test_auc_macro", "Macro AUC")]
    w = 0.8 / len(metrics)
    fig, ax = plt.subplots(figsize=(max(8.6, 1.15 * len(df)), 5.6))
    for i, (key, label) in enumerate(metrics):
        ax.bar(x + (i - (len(metrics) - 1) / 2) * w, df[key], w, label=label, color=OKABE_ITO[i],
               edgecolor="black", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_xlim(-0.6, len(df) - 0.4)
    ax.set_ylim(0, 1.15)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_ylabel("Score (test set)")
    ax.set_title("Test-set performance after tuning")
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(ncol=4, fontsize=11, loc="upper center")
    save_fig(fig, FIG_SUM / "test_metrics")

    fig, ax = plt.subplots(figsize=(max(8.6, 1.15 * len(df)), 5.6))
    colors = [KIND_COLORS.get(k, "#7F7F7F") for k in df["kind"]]
    bars = ax.bar(x, df["fit_time_min"], 0.6, color=colors, edgecolor="black", linewidth=1.0)
    for b, v in zip(bars, df["fit_time_min"]):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("Training time (min)")
    ax.set_title("Training time of the reported model")
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    save_fig(fig, FIG_SUM / "training_time")

    grids = [(n, pd.read_csv(OUT_DIR / n / "grid_search.csv")) for n in TUNED_ORDER
             if (OUT_DIR / n / "grid_search.csv").exists()]
    if grids:
        fig, axes = plt.subplots(1, len(grids), figsize=(6.2 * len(grids), 5.4), squeeze=False)
        for ax, (n, g) in zip(axes[0], grids):
            table = g.pivot_table(index=["learning_rate", "dropout_rate"], columns=["optimizer", "batch_size"],
                                  values="val_f1_macro")
            sns.heatmap(table, annot=True, fmt=".3f", cmap="Blues", linewidths=0.6, linecolor="white",
                        annot_kws={"size": 10, "weight": "bold"}, cbar_kws={"label": "val macro F1"}, ax=ax)
            ax.minorticks_off()
            ax.set_title(n)
            ax.set_xlabel("optimizer, batch size")
            ax.set_ylabel("learning rate, dropout")
        fig.suptitle("Grid search: validation macro F1", fontweight="bold", y=1.03)
        save_fig(fig, FIG_SUM / "grid_heatmaps")

### 8.3 Reading the results
- **Tuned models** are fine-tuned end to end, so they should land far above the frozen screening numbers (0.49–0.61 macro F1) and near Custom-ENB5's 0.89 test accuracy.
- **The grid tables** (`grid_table_val_f1_macro.csv`, `grid_table_val_accuracy.csv` per model) have the same shape as the existing parameter-tuning table: learning rate × dropout down the rows, optimizer × batch size across the columns.
- **Validation numbers** come from the restored best-epoch weights. The **test set** is evaluated once per model, on the winning combination only, so the grid never sees it.
- **Training time** is the fit time of the reported model, not of the whole grid. Per-combination times are in `grid_search.csv`.
- **Classical baselines** use flattened pixels, so they are not comparable in kind to the fine-tuned networks; they set the "no deep learning" floor.